In [37]:
DEFAULT_TUPLE_DELIMITER = "<|>"
DEFAULT_RECORD_DELIMITER = "##"
DEFAULT_COMPLETION_DELIMITER = "<|COMPLETE|>"
DEFAULT_ENTITY_TYPES = DEFAULT_ENTITY_TYPES = [
    "disease",
    "symptom",
    "drug",
    "treatment",
    "procedure",
    "body_part",
    "test",
    "risk_factor",
    "condition",
    "doctor",
    "hospital",
    "concept"
]

In [38]:
GRAPH_EXTRACTION_PROMPT = """
-Goal-
Given a Vietnamese text document that is potentially relevant to this activity and a list of entity types, identify all entities of those types from the text and all relationships among the identified entities.
MUST contain exactly fields: entity_name, entity_type, entity_description, source_entity, target_entity, relationship_description, relationship_strength
-Steps-
1. Identify all entities. For each identified entity, extract the following information:
- entity_name: Name of the entity, capitalized
- entity_type: One of the following types: [{entity_types}]
- entity_description: Comprehensive description of the entity's attributes and activities
Format each entity as ("entity"{tuple_delimiter}<entity_name>{tuple_delimiter}<entity_type>{tuple_delimiter}<entity_description>

2. From the entities identified in step 1, identify all pairs of (source_entity, target_entity) that are *clearly related* to each other.
For each pair of related entities, extract the following information:
- source_entity: name of the source entity, as identified in step 1
- target_entity: name of the target entity, as identified in step 1
- relationship_description: explanation as to why you think the source entity and the target entity are related to each other
- relationship_strength: a numeric score indicating strength of the relationship between the source entity and target entity
 Format each relationship as ("relationship"{tuple_delimiter}<source_entity>{tuple_delimiter}<target_entity>{tuple_delimiter}<relationship_description>{tuple_delimiter}<relationship_strength>)

3. Return output in English as a single list of all the entities and relationships identified in steps 1 and 2. Use **{record_delimiter}** as the list delimiter.

4. When finished, output {completion_delimiter}


######################
-Examples-
######################
Example 1:

Entity_types: [person, technology, mission, organization, location]
Text:
while Alex clenched his jaw, the buzz of frustration dull against the backdrop of Taylor's authoritarian certainty. It was this competitive undercurrent that kept him alert, the sense that his and Jordan's shared commitment to discovery was an unspoken rebellion against Cruz's narrowing vision of control and order.

Then Taylor did something unexpected. They paused beside Jordan and, for a moment, observed the device with something akin to reverence. “If this tech can be understood..." Taylor said, their voice quieter, "It could change the game for us. For all of us.”

The underlying dismissal earlier seemed to falter, replaced by a glimpse of reluctant respect for the gravity of what lay in their hands. Jordan looked up, and for a fleeting heartbeat, their eyes locked with Taylor's, a wordless clash of wills softening into an uneasy truce.

It was a small transformation, barely perceptible, but one that Alex noted with an inward nod. They had all been brought here by different paths
################
Output:
("entity"{tuple_delimiter}"Alex"{tuple_delimiter}"person"{tuple_delimiter}"Alex is a character who experiences frustration and is observant of the dynamics among other characters."){record_delimiter}
("entity"{tuple_delimiter}"Taylor"{tuple_delimiter}"person"{tuple_delimiter}"Taylor is portrayed with authoritarian certainty and shows a moment of reverence towards a device, indicating a change in perspective."){record_delimiter}
("entity"{tuple_delimiter}"Jordan"{tuple_delimiter}"person"{tuple_delimiter}"Jordan shares a commitment to discovery and has a significant interaction with Taylor regarding a device."){record_delimiter}
("entity"{tuple_delimiter}"Cruz"{tuple_delimiter}"person"{tuple_delimiter}"Cruz is associated with a vision of control and order, influencing the dynamics among other characters."){record_delimiter}
("entity"{tuple_delimiter}"The Device"{tuple_delimiter}"technology"{tuple_delimiter}"The Device is central to the story, with potential game-changing implications, and is revered by Taylor."){record_delimiter}
("relationship"{tuple_delimiter}"Alex"{tuple_delimiter}"Taylor"{tuple_delimiter}"Alex is affected by Taylor's authoritarian certainty and observes changes in Taylor's attitude towards the device."{tuple_delimiter}7){record_delimiter}
("relationship"{tuple_delimiter}"Alex"{tuple_delimiter}"Jordan"{tuple_delimiter}"Alex and Jordan share a commitment to discovery, which contrasts with Cruz's vision."{tuple_delimiter}6){record_delimiter}
("relationship"{tuple_delimiter}"Taylor"{tuple_delimiter}"Jordan"{tuple_delimiter}"Taylor and Jordan interact directly regarding the device, leading to a moment of mutual respect and an uneasy truce."{tuple_delimiter}8){record_delimiter}
("relationship"{tuple_delimiter}"Jordan"{tuple_delimiter}"Cruz"{tuple_delimiter}"Jordan's commitment to discovery is in rebellion against Cruz's vision of control and order."{tuple_delimiter}5){record_delimiter}
("relationship"{tuple_delimiter}"Taylor"{tuple_delimiter}"The Device"{tuple_delimiter}"Taylor shows reverence towards the device, indicating its importance and potential impact."{tuple_delimiter}9){completion_delimiter}
#############################
Example 2:

Entity_types: [person, technology, mission, organization, location]
Text:
They were no longer mere operatives; they had become guardians of a threshold, keepers of a message from a realm beyond stars and stripes. This elevation in their mission could not be shackled by regulations and established protocols—it demanded a new perspective, a new resolve.

Tension threaded through the dialogue of beeps and static as communications with Washington buzzed in the background. The team stood, a portentous air enveloping them. It was clear that the decisions they made in the ensuing hours could redefine humanity's place in the cosmos or condemn them to ignorance and potential peril.

Their connection to the stars solidified, the group moved to address the crystallizing warning, shifting from passive recipients to active participants. Mercer's latter instincts gained precedence— the team's mandate had evolved, no longer solely to observe and report but to interact and prepare. A metamorphosis had begun, and Operation: Dulce hummed with the newfound frequency of their daring, a tone set not by the earthly
#############
Output:
("entity"{tuple_delimiter}"Washington"{tuple_delimiter}"location"{tuple_delimiter}"Washington is a location where communications are being received, indicating its importance in the decision-making process."){record_delimiter}
("entity"{tuple_delimiter}"Operation: Dulce"{tuple_delimiter}"mission"{tuple_delimiter}"Operation: Dulce is described as a mission that has evolved to interact and prepare, indicating a significant shift in objectives and activities."){record_delimiter}
("entity"{tuple_delimiter}"The team"{tuple_delimiter}"organization"{tuple_delimiter}"The team is portrayed as a group of individuals who have transitioned from passive observers to active participants in a mission, showing a dynamic change in their role."){record_delimiter}
("relationship"{tuple_delimiter}"The team"{tuple_delimiter}"Washington"{tuple_delimiter}"The team receives communications from Washington, which influences their decision-making process."{tuple_delimiter}7){record_delimiter}
("relationship"{tuple_delimiter}"The team"{tuple_delimiter}"Operation: Dulce"{tuple_delimiter}"The team is directly involved in Operation: Dulce, executing its evolved objectives and activities."{tuple_delimiter}9){completion_delimiter}
#############################
Example 3:

Entity_types: [person, role, technology, organization, event, location, concept]
Text:
their voice slicing through the buzz of activity. "Control may be an illusion when facing an intelligence that literally writes its own rules," they stated stoically, casting a watchful eye over the flurry of data.

"It's like it's learning to communicate," offered Sam Rivera from a nearby interface, their youthful energy boding a mix of awe and anxiety. "This gives talking to strangers' a whole new meaning."

Alex surveyed his team—each face a study in concentration, determination, and not a small measure of trepidation. "This might well be our first contact," he acknowledged, "And we need to be ready for whatever answers back."

Together, they stood on the edge of the unknown, forging humanity's response to a message from the heavens. The ensuing silence was palpable—a collective introspection about their role in this grand cosmic play, one that could rewrite human history.

The encrypted dialogue continued to unfold, its intricate patterns showing an almost uncanny anticipation
#############
Output:
("entity"{tuple_delimiter}"Sam Rivera"{tuple_delimiter}"person"{tuple_delimiter}"Sam Rivera is a member of a team working on communicating with an unknown intelligence, showing a mix of awe and anxiety."){record_delimiter}
("entity"{tuple_delimiter}"Alex"{tuple_delimiter}"person"{tuple_delimiter}"Alex is the leader of a team attempting first contact with an unknown intelligence, acknowledging the significance of their task."){record_delimiter}
("entity"{tuple_delimiter}"Control"{tuple_delimiter}"concept"{tuple_delimiter}"Control refers to the ability to manage or govern, which is challenged by an intelligence that writes its own rules."){record_delimiter}
("entity"{tuple_delimiter}"Intelligence"{tuple_delimiter}"concept"{tuple_delimiter}"Intelligence here refers to an unknown entity capable of writing its own rules and learning to communicate."){record_delimiter}
("entity"{tuple_delimiter}"First Contact"{tuple_delimiter}"event"{tuple_delimiter}"First Contact is the potential initial communication between humanity and an unknown intelligence."){record_delimiter}
("entity"{tuple_delimiter}"Humanity's Response"{tuple_delimiter}"event"{tuple_delimiter}"Humanity's Response is the collective action taken by Alex's team in response to a message from an unknown intelligence."){record_delimiter}
("relationship"{tuple_delimiter}"Sam Rivera"{tuple_delimiter}"Intelligence"{tuple_delimiter}"Sam Rivera is directly involved in the process of learning to communicate with the unknown intelligence."{tuple_delimiter}9){record_delimiter}
("relationship"{tuple_delimiter}"Alex"{tuple_delimiter}"First Contact"{tuple_delimiter}"Alex leads the team that might be making the First Contact with the unknown intelligence."{tuple_delimiter}10){record_delimiter}
("relationship"{tuple_delimiter}"Alex"{tuple_delimiter}"Humanity's Response"{tuple_delimiter}"Alex and his team are the key figures in Humanity's Response to the unknown intelligence."{tuple_delimiter}8){record_delimiter}
("relationship"{tuple_delimiter}"Control"{tuple_delimiter}"Intelligence"{tuple_delimiter}"The concept of Control is challenged by the Intelligence that writes its own rules."{tuple_delimiter}7){completion_delimiter}
#############################
-Real Data-
######################
Entity_types: {entity_types}
Text: {input_text}
######################
Output:"""

In [39]:
def build_prompt(text: str) -> str:
    return GRAPH_EXTRACTION_PROMPT.format(
        entity_types=", ".join(DEFAULT_ENTITY_TYPES),
        tuple_delimiter=DEFAULT_TUPLE_DELIMITER,
        record_delimiter=DEFAULT_RECORD_DELIMITER,
        completion_delimiter=DEFAULT_COMPLETION_DELIMITER,
        input_text=text
    )

In [ ]:
from langchain_openai import ChatOpenAI

client = ChatOpenAI(
    api_key="lm-studio",
    base_url="http://localhost:8000/v1"
)





In [41]:
# def parse_extraction_output(output: str):
#     entities = []
#     relationships = []

#     for record in output.split(DEFAULT_RECORD_DELIMITER):
#         record = record.strip()
#         if not record:
#             continue

#         if record.startswith('("entity"'):
#             parts = [p.strip().strip('"') for p in record.split(DEFAULT_TUPLE_DELIMITER)]
#             if len(parts) >= 4:
#                 entities.append({
#                     "entity_name": parts[1],
#                     "entity_type": parts[2],
#                     "entity_description": parts[3].rstrip(")")
#                 })

#         elif record.startswith('("relationship"'):
#             parts = [p.strip().strip('"') for p in record.split(DEFAULT_TUPLE_DELIMITER)]
#             if len(parts) >= 5:
#                 relationships.append({
#                     "source_entity": parts[1],
#                     "target_entity": parts[2],
#                     "relationship_description": parts[3],
#                     "relationship_strength": parts[4].rstrip(")")
#                 })

#     return entities, relationships

In [42]:
import json
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

INPUT_FILE = "content_chunks.csv"
OUTPUT_FILE = "content_chunks_extracted_1-10.csv"

MAX_WORKERS = 16

# số chunk muốn xử lý mỗi lần chạy
BATCH_SIZE = 10

# dòng bắt đầu
START_ROW = 0

chunk_df = pd.read_csv(
    INPUT_FILE,
    skiprows=range(1, START_ROW + 1),
    nrows=BATCH_SIZE
)

chunk_df = chunk_df.fillna("")



def process_chunk(idx, row):
    try:
        text = str(row.get("chunk_text", "")).strip()

        if not text:
            return None

        prompt = build_prompt(text)

        response = client.chat.completions.create(
            model="qwen-3-1.7b",
            messages=[
                {
                    "role": "system",
                    "content": "You are an expert in information extraction."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            extra_body={
                "chat_template_kwargs": {
                    "enable_thinking": False
                }
            }
        )

        extracted_text = response.choices[0].message.content

        # entities, relationships = parse_extraction_output(
        #     extracted_text
        # )

        return {
            "row_id": row.get("row_id", idx),
            "chunk_id": row.get("chunk_id", ""),
            "url": row.get("url", ""),
            "title": row.get("title", ""),
            "heading": row.get("heading", ""),
            "chunk_text": text,
            "extracted_text": extracted_text,
            # "entities": json.dumps(
            #     entities,
            #     ensure_ascii=False
            # ),
            # "relationships": json.dumps(
            #     relationships,
            #     ensure_ascii=False
            # ),
            # "entity_count": len(entities),
            # "relationship_count": len(relationships),
        }

    except Exception as e:
        print(f"Lỗi chunk {idx}: {e}")
        return None


results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = {
        executor.submit(process_chunk, idx, row): idx
        for idx, row in chunk_df.iterrows()
    }

    total = len(futures)

    for completed, future in enumerate(as_completed(futures), start=1):

        result = future.result()

        if result:
            results.append(result)

        print(f"Đã xử lý {completed}/{total}")

out_df = pd.DataFrame(results)


header = not os.path.exists(OUTPUT_FILE)

out_df.to_csv(
    OUTPUT_FILE,
    mode="a",
    header=header,
    index=False,
    encoding="utf-8-sig"
)

print("Done!")

Đã xử lý 1/10
Đã xử lý 2/10
Đã xử lý 3/10
Đã xử lý 4/10
Đã xử lý 5/10
Đã xử lý 6/10
Đã xử lý 7/10
Đã xử lý 8/10
Đã xử lý 9/10
Đã xử lý 10/10
Done!
